# Portfolio Researcher API demo (quant_api.portfolio)

Decision note DEC-017: the Portfolio Researcher loads the research pool, orthogonalizes new candidates against it (Component 3 - Orthogonalization), combines scores into a composite (Component 4 - Combination), refits allocation weights on a cadence (handoff artifact `weights.json` consumed by the live `PortfolioConfig`), and produces the monthly portfolio health report.

This notebook runs from the repo root with the project `.venv` (which has `src/` on `sys.path`). Pool files are written under a `tempfile.mkdtemp()` folder; scoring uses the full VN30F1M 1-minute history from the research catalog (~474k bars, ~1-2 s per cell), so the metrics below are real, not illustrative.

In [ ]:
import json, tempfile, pathlib
import quant_api.portfolio as pf
from quant_api.core.pool import PoolEntry, write_pool_entry, write_pool_index

# Build a throwaway pool folder (3 alphas). The pool is normally written
# by the research role; the folder is the single source of truth.
tmp = tempfile.mkdtemp(prefix="pool_demo_")
entries = [
    PoolEntry(alpha_id="mom_8", dsl="ts_returns(close, 8)"),
    PoolEntry(alpha_id="rev_5", dsl="-ts_returns(close, 5)"),
    PoolEntry(alpha_id="dev_8", dsl="close - ewma(close, 8)"),
]
for e in entries:
    write_pool_entry(e, root=tmp)
write_pool_index(entries, root=tmp)
print("pool folder:", tmp)
print("loaded pool:", [(e.alpha_id, e.dsl) for e in pf.load_pool(root=tmp)])

In [ ]:
# Score the whole pool in ONE engine batch call, then combine (inverse_vol).
scores = pf.pool_scores(pf.load_pool(root=tmp))
print("score series:", {k: len(v) for k, v in scores.items()})

combo = pf.combine(scores, method="inverse_vol")
print("method:", combo["method"], "provenance:", combo["provenance"])
print("weights:", {k: round(w, 4) for k, w in combo["weights"].items()})
print("composite head:", [round(x, 4) for x in combo["composite"][:5]])

In [ ]:
# Weekly cadence: refit weights -> weights.json handoff artifact.
refit = pf.refit_weights(pf.load_pool(root=tmp), method="inverse_vol", root=tmp)
print("refit:", refit["generated"], refit["method"], refit["provenance"])
print("weights:", {k: round(w, 4) for k, w in refit["weights"].items()})
saved = json.loads(pathlib.Path(tmp, "weights.json").read_text())
assert saved == refit, "weights.json must mirror the returned payload"
print("weights.json written ->", pathlib.Path(tmp, "weights.json"))

In [ ]:
# Monthly cadence: portfolio health report (in-memory, never auto-saved).
report = pf.portfolio_health_report(pf.load_pool(root=tmp), window_days=30)
for k, v in report.items():
    if isinstance(v, float):
        print(f"{k}: {round(v, 4)}")
    else:
        print(f"{k}: {v}")

In [ ]:
# Component 3 - Orthogonalization: is a candidate additive to the pool?
# An in-span candidate (exact copy of a pool member) is fully absorbed.
print("in-span:", pf.orthogonalize(scores["dev_8"], [scores["dev_8"]]))
# A genuinely new candidate vs the rest of the pool gets a verdict.
rest = [scores[k] for k in scores if k != "mom_8"]
print("candidate mom_8 vs rest:", pf.orthogonalize(scores["mom_8"], rest))